# 🏭 Applied Vision Transformer Report: Multi-Category Industrial Defect Detection on MVTec AD

**Goal:** Detect manufacturing defects across **multiple MVTec AD categories** by comparing:
- **Approach A – CLIP Zero-Shot:** No training. Uses text prompts to score images.
- **Approach B – ViT + LoRA:** Supervised fine-tuning with parameter-efficient LoRA per category.

**Dataset:** MVTec AD (via FiftyOne Hub) – 15 categories (10 objects + 5 textures).

**What's new in this version:**
- Loop over multiple categories automatically
- Per-category AUROC comparison table
- Cross-category summary plots

⚠️ **First:** `Runtime > Change runtime type > T4 GPU`, then run cells top-to-bottom.

---
## 📦 Step 0: Install Libraries & Mount Drive

In [ ]:
#@title Step 0a: Install libraries
!pip install -q fiftyone transformers datasets evaluate accelerate peft pillow matplotlib scikit-learn "torchao>=0.16.0"

In [ ]:
#@title Step 0b: Mount Google Drive (recommended for checkpoint saving)
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/MVTec_ViT_Project'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Project directory ready: {PROJECT_DIR}')

---
## 📊 Step 1: Load Dataset, Discover Categories & Build Splits

MVTec AD contains **15 categories**. We load the full dataset once, then filter per category.
- `category.label` = object/texture type (e.g., `'bottle'`, `'carpet'`)
- `defect.label` = defect type (`'good'` or a defect name like `'broken_large'`)

**Split strategy (same for all categories):**
- **ViT training set** = all `train/good` images + 50% of defective test images
- **Evaluation set** = remaining 50% of defective test images + all `test/good` images
- **Both CLIP and ViT evaluated on identical eval set** → fair AUROC comparison.

In [ ]:
#@title Step 1a: Load full MVTec AD dataset and discover categories
import fiftyone as fo
import fiftyone.utils.huggingface as fouh
from fiftyone import ViewField as F
import random
import numpy as np

# --- Load the full dataset from the Hub (one-time) ---
dataset = fouh.load_from_hub('Voxel51/mvtec-ad')

# --- Discover all categories ---
ALL_CATEGORIES = sorted(dataset.distinct('category.label'))
print(f'Total categories in MVTec AD: {len(ALL_CATEGORIES)}')
print('Categories:', ALL_CATEGORIES)

# Show sample counts per category
counts = dataset.count_values('category.label')
for cat in ALL_CATEGORIES:
    print(f'  {cat}: {counts[cat]} samples')

In [ ]:
#@title Step 1b: Configure which categories to run
# ---------------------------------------------------------------
# CHOOSE YOUR CATEGORIES HERE
# Option A: Run ALL 15 categories (slow, ~30-60 min each for ViT training)
# SELECTED_CATEGORIES = ALL_CATEGORIES
#
# Option B: Run a representative subset (recommended for Colab)
SELECTED_CATEGORIES = [
    'bottle',      # original category from your notebook
    'cable',       # long/thin object, different from bottle
    'capsule',     # small pharmaceutical object
    'hazelnut',    # organic shape, textured surface
    'metal_nut',   # metallic, reflective surface
    'pill',        # small, round, multiple defect types
    'screw',       # thin, thread details
    'toothbrush',  # elongated, plastic
    'transistor',  # electronic component, complex shape
    'zipper',      # mechanical, textured
    'carpet',      # texture category (not object)
    'grid',        # regular pattern texture
    'leather',     # natural texture variation
    'tile',        # ceramic, regular pattern
    'wood'         # natural grain texture
]
# ---------------------------------------------------------------

print(f'Will run on {len(SELECTED_CATEGORIES)} categories:')
for i, cat in enumerate(SELECTED_CATEGORIES, 1):
    print(f'  {i}. {cat}')

In [ ]:
#@title Step 1c: Helper function to build train/eval splits for any category
def build_splits(category_name, seed=42):
    """
    Given a category name (e.g., 'bottle'), build train/eval views.
    Returns: (train_view, eval_view, eval_labels)
    """
    cat_ds = dataset.match(F('category.label') == category_name)

    def get_split(ds, split_name):
        try:
            if 'split' in ds.get_field_schema():
                v = ds.match(F('split') == split_name)
                if len(v) > 0:
                    return v
        except Exception:
            pass
        try:
            v = ds.match_tags(split_name)
            if len(v) > 0:
                return v
        except Exception:
            pass
        return ds.match(F('filepath').contains(split_name))

    train_split = get_split(cat_ds, 'train')
    test_split  = get_split(cat_ds, 'test')

    train_good_ids = [s.id for s in train_split if s.defect.label == 'good']
    test_good_ids  = [s.id for s in test_split  if s.defect.label == 'good']
    test_bad_ids   = [s.id for s in test_split  if s.defect.label != 'good']

    # 50/50 split of defective test images
    random.seed(seed)
    random.shuffle(test_bad_ids)
    mid = len(test_bad_ids) // 2
    bad_train_ids = test_bad_ids[:mid]
    bad_eval_ids  = test_bad_ids[mid:]

    train_ids = train_good_ids + bad_train_ids
    eval_ids  = bad_eval_ids + test_good_ids

    train_view = cat_ds.select(train_ids)
    eval_view  = cat_ds.select(eval_ids)
    eval_labels = np.array([0 if s.defect.label == 'good' else 1 for s in eval_view])

    return train_view, eval_view, eval_labels

# Quick test on bottle to verify
t, e, l = build_splits('bottle')
print(f'Bottle — Train: {len(t)} | Eval: {len(e)} (defective: {int(l.sum())}, good: {int((l==0).sum())})')

---
## 🧠 Step 2: CLIP Zero-Shot (Category-Aware Prompts)

We load CLIP once, then loop over categories using category-specific text prompts.

In [ ]:
#@title Step 2a: Load CLIP model
from transformers import CLIPProcessor, CLIPModel
import torch

model_id = 'openai/clip-vit-base-patch32'
clip_model = CLIPModel.from_pretrained(model_id).to('cuda').eval()
clip_processor = CLIPProcessor.from_pretrained(model_id)
print('CLIP loaded.')

In [ ]:
#@title Step 2b: CLIP inference function (category-aware prompts)
from tqdm import tqdm
import numpy as np
from PIL import Image

def get_clip_scores_for_category(eval_view, category_name):
    """
    Compute CLIP anomaly scores for a given category using category-specific prompts.
    Score = logit(defective) - logit(normal)
    """
    # Category-aware prompts (handle multi-word categories like 'metal_nut' -> 'metal nut')
    display_name = category_name.replace('_', ' ')
    text_prompts = [
        f'a photo of a normal {display_name}',
        f'a photo of a defective {display_name}'
    ]
    scores = []
    for s in tqdm(eval_view, desc=f'CLIP-{category_name}', leave=False):
        image = Image.open(s.filepath).convert('RGB')
        inputs = clip_processor(text=text_prompts, images=image, return_tensors='pt', padding=True).to('cuda')
        with torch.no_grad():
            logits = clip_model(**inputs).logits_per_image.squeeze(0)
        scores.append((logits[1] - logits[0]).item())
    return np.array(scores)

# Test on bottle
_, bottle_eval, bottle_labels = build_splits('bottle')
bottle_clip = get_clip_scores_for_category(bottle_eval, 'bottle')
print(f'Bottle CLIP scores computed: {len(bottle_clip)} samples')

In [ ]:
#@title Step 2c: Run CLIP on ALL selected categories
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import MinMaxScaler

clip_results = {}  # category -> {scores, labels, auroc}

print('='*60)
print('Running CLIP Zero-Shot across selected categories...')
print('='*60)

for cat in SELECTED_CATEGORIES:
    _, eval_view, eval_labels = build_splits(cat)
    scores = get_clip_scores_for_category(eval_view, cat)
    scores_norm = MinMaxScaler().fit_transform(scores.reshape(-1, 1)).flatten()
    auroc = roc_auc_score(eval_labels, scores_norm)
    clip_results[cat] = {
        'scores': scores_norm,
        'labels': eval_labels,
        'auroc': auroc,
        'n_eval': len(eval_labels),
        'n_defective': int(eval_labels.sum()),
        'n_good': int((eval_labels == 0).sum())
    }
    print(f'{cat:15s} | AUROC: {auroc:.4f} | Eval: {len(eval_labels)} (bad:{int(eval_labels.sum())}, good:{int((eval_labels==0).sum())})')

print('
' + '='*60)
print('CLIP Zero-Shot Summary:')
print(f'  Mean AUROC: {np.mean([v["auroc"] for v in clip_results.values()]):.4f}')
print(f'  Std  AUROC: {np.std([v["auroc"] for v in clip_results.values()]):.4f}')

---
## 🧠 Step 3: ViT + LoRA Fine-Tuning (Per Category)

We train a separate LoRA adapter for each category. Each adapter is saved to Drive for later use.

In [ ]:
#@title Step 3a: PyTorch Dataset wrapper (reusable across categories)
from torch.utils.data import Dataset as TorchDataset, DataLoader
from transformers import AutoImageProcessor
from PIL import Image
import torch

vit_processor = AutoImageProcessor.from_pretrained('google/vit-base-patch16-224')

class FiftyOneViewDataset(TorchDataset):
    def __init__(self, view, processor):
        self.samples = list(view)
        self.processor = processor
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s.filepath).convert('RGB')
        inputs = self.processor(images=img, return_tensors='pt')
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs['labels'] = torch.tensor(0 if s.defect.label == 'good' else 1, dtype=torch.long)
        return inputs

In [ ]:
#@title Step 3b: Helper to build ViT+LoRA model
from transformers import ViTForImageClassification
from peft import LoraConfig, get_peft_model
import torch

def build_vit_lora_model(num_labels=2):
    """Build a fresh ViT + LoRA model."""
    model = ViTForImageClassification.from_pretrained(
        'google/vit-base-patch16-224',
        num_labels=num_labels,
        ignore_mismatched_sizes=True,
    )
    # Discover target modules
    linear_names = [n for n, m in model.named_modules() if isinstance(m, torch.nn.Linear)]
    target_modules = [n for n in linear_names if n.endswith('query') or n.endswith('value')]
    if not target_modules:
        target_modules = linear_names
    lora_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=target_modules,
        lora_dropout=0.1, bias='none',
        modules_to_save=['classifier'],
    )
    model = get_peft_model(model, lora_config)
    return model.to('cuda')

In [ ]:
#@title Step 3c: Training function per category
from transformers import TrainingArguments, Trainer
import evaluate, numpy as np

accuracy_metric = evaluate.load('accuracy')
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    return accuracy_metric.compute(predictions=np.argmax(preds, axis=1), references=labels)

def train_vit_for_category(category_name, train_view, eval_view, epochs=5, batch_size=16):
    """Train ViT+LoRA for one category and return the trained model + trainer."""
    train_ds = FiftyOneViewDataset(train_view, vit_processor)
    eval_ds  = FiftyOneViewDataset(eval_view,  vit_processor)

    model = build_vit_lora_model()

    save_dir = f'{PROJECT_DIR}/vit_lora_{category_name}'
    training_args = TrainingArguments(
        output_dir=save_dir,
        learning_rate=2e-4,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        weight_decay=0.01,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='accuracy',
        report_to='none',
        fp16=True,
        remove_unused_columns=False,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    model.save_pretrained(f'{save_dir}_final')
    print(f'Saved adapter to: {save_dir}_final')
    return model, trainer

print('Training helper ready.')

In [ ]:
#@title 🔧 Fix torchao compatibility issue (run before training)
# PEFT checks for torchao and throws an error if an old version is installed.
# This cell upgrades it so training works. Run once per session.
!pip install -q "torchao>=0.16.0"
print("torchao fixed. You can now run Step 3d.")

In [ ]:
#@title Step 3d: Run ViT+LoRA on ALL selected categories

vit_results = {}  # category -> {scores, labels, auroc}

print('='*60)
print('Training ViT + LoRA across selected categories...')
print('(This will take time. Each category ~3-5 min on T4)')
print('='*60)

for cat in SELECTED_CATEGORIES:
    print(f'
>>> Category: {cat}')
    train_view, eval_view, eval_labels = build_splits(cat)
    # --- Uncomment to train ---
    # model_vit, trainer = train_vit_for_category(cat, train_view, eval_view, epochs=5)
    # --------------------------
    # --- OR: Load previously trained adapter ---
    # from peft import PeftModel
    # base = build_vit_lora_model()
    # model_vit = PeftModel.from_pretrained(base, f'{PROJECT_DIR}/vit_lora_{cat}_final')
    # model_vit = model_vit.to('cuda')
    # ------------------------------------------

    # Inference
    eval_ds = FiftyOneViewDataset(eval_view, vit_processor)
    loader = DataLoader(eval_ds, batch_size=16, shuffle=False)
    model_vit.eval()
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            b = {k: v.to('cuda') for k, v in batch.items() if k != 'labels'}
            logits = model_vit(**b).logits
            all_probs.append(torch.softmax(logits, dim=-1)[:, 1].cpu())
    vit_scores = torch.cat(all_probs).numpy()

    auroc = roc_auc_score(eval_labels, vit_scores)
    vit_results[cat] = {
        'scores': vit_scores,
        'labels': eval_labels,
        'auroc': auroc
    }
    print(f'{cat:15s} | AUROC: {auroc:.4f}')

print('
' + '='*60)
print('ViT + LoRA Summary:')
print(f'  Mean AUROC: {np.mean([v["auroc"] for v in vit_results.values()]):.4f}')
print(f'  Std  AUROC: {np.std([v["auroc"] for v in vit_results.values()]):.4f}')

---
## 📈 Step 4: Cross-Category AUROC Comparison

In [ ]:
#@title Step 4a: Build comparison table
import pandas as pd

rows = []
for cat in SELECTED_CATEGORIES:
    rows.append({
        'Category': cat,
        'CLIP_AUROC': clip_results[cat]['auroc'],
        'ViT_AUROC': vit_results[cat]['auroc'],
        'Delta (ViT - CLIP)': vit_results[cat]['auroc'] - clip_results[cat]['auroc'],
        'N_eval': clip_results[cat]['n_eval'],
        'N_defective': clip_results[cat]['n_defective'],
        'N_good': clip_results[cat]['n_good']
    })

df = pd.DataFrame(rows)
print('
=== Per-Category AUROC Comparison ===')
print(df.to_string(index=False))

print(f'
Mean CLIP AUROC: {df["CLIP_AUROC"].mean():.4f}')
print(f'Mean ViT  AUROC: {df["ViT_AUROC"].mean():.4f}')
print(f'Mean Delta:      {df["Delta (ViT - CLIP)"].mean():.4f}')

In [ ]:
#@title Step 4b: Bar chart comparison
import matplotlib.pyplot as plt

x = np.arange(len(SELECTED_CATEGORIES))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, [clip_results[c]['auroc'] for c in SELECTED_CATEGORIES], width, label='CLIP Zero-Shot', color='steelblue')
bars2 = ax.bar(x + width/2, [vit_results[c]['auroc'] for c in SELECTED_CATEGORIES], width, label='ViT + LoRA', color='coral')

ax.set_ylabel('AUROC')
ax.set_title('AUROC Comparison: CLIP Zero-Shot vs ViT + LoRA (per category)')
ax.set_xticks(x)
ax.set_xticklabels(SELECTED_CATEGORIES, rotation=45, ha='right')
ax.legend()
ax.set_ylim([0.5, 1.0])
ax.axhline(0.95, color='green', linestyle='--', alpha=0.5, label='SOTA reference (0.95)')
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=7)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
#@title Step 4c: Delta plot (ViT advantage per category)
deltas = [vit_results[c]['auroc'] - clip_results[c]['auroc'] for c in SELECTED_CATEGORIES]
colors = ['green' if d > 0 else 'red' for d in deltas]

plt.figure(figsize=(12, 5))
bars = plt.bar(SELECTED_CATEGORIES, deltas, color=colors, alpha=0.7)
plt.axhline(0, color='black', linewidth=0.8)
plt.ylabel('AUROC Difference (ViT + LoRA − CLIP)')
plt.title('Per-Category Improvement of ViT + LoRA over CLIP Zero-Shot')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

for bar, d in zip(bars, deltas):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (0.01 if d > 0 else -0.02),
             f'{d:+.3f}', ha='center', va='bottom' if d > 0 else 'top', fontsize=8)

plt.tight_layout()
plt.show()

---
## 📉 Step 5: ROC Curves (Sample Categories)

In [ ]:
#@title Plot ROC curves for first 4 categories
from sklearn.metrics import roc_curve

sample_cats = SELECTED_CATEGORIES[:4]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, cat in zip(axes, sample_cats):
    fpr_c, tpr_c, _ = roc_curve(clip_results[cat]['labels'], clip_results[cat]['scores'])
    fpr_v, tpr_v, _ = roc_curve(vit_results[cat]['labels'], vit_results[cat]['scores'])
    ax.plot(fpr_c, tpr_c, label=f"CLIP (AUC={clip_results[cat]['auroc']:.3f})")
    ax.plot(fpr_v, tpr_v, label=f"ViT+LoRA (AUC={vit_results[cat]['auroc']:.3f})")
    ax.plot([0,1], [0,1], 'k--', alpha=0.5)
    ax.set_title(f'{cat}')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

plt.suptitle('ROC Curves – CLIP vs ViT+LoRA', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 💾 Step 6: Save All Results to Drive

In [ ]:
#@title Save consolidated results
import json, pickle

# Save summary table
csv_path = f'{PROJECT_DIR}/multi_category_auroc_results.csv'
df.to_csv(csv_path, index=False)
print(f'Saved CSV: {csv_path}')

# Save detailed results
results_dict = {
    'clip': {k: {'auroc': v['auroc'], 'n_eval': v['n_eval']} for k, v in clip_results.items()},
    'vit':  {k: {'auroc': v['auroc']} for k, v in vit_results.items()},
    'categories': SELECTED_CATEGORIES
}
json_path = f'{PROJECT_DIR}/multi_category_results.json'
with open(json_path, 'w') as f:
    json.dump(results_dict, f, indent=2)
print(f'Saved JSON: {json_path}')

print('
✅ All done! Download the notebook via File > Download > Download .ipynb')

---
## 📝 Step 7: Analysis & Report Writing Guide

### 7.1 Performance Summary
- **Tabulate** `CLIP_AUROC` vs `ViT_AUROC` per category from the CSV.
- **Which categories favor ViT+LoRA?** Usually object categories with consistent backgrounds (bottle, transistor) benefit most from supervised adaptation.
- **Which categories favor CLIP?** Texture categories (carpet, tile, wood) or highly variable objects where zero-shot semantic understanding is strong.
- **Cross-category variance:** A large std in ViT AUROC suggests the model struggles to generalise across object types—this is expected since each adapter only sees one category.

### 7.2 Bias Analysis
- **Dataset bias:** MVTec AD images are studio-quality with fixed backgrounds. Real factory lines have variable lighting, dust, and camera angles. Report this as the primary limitation.
- **CLIP prompt bias:** The prompt `a photo of a defective X` may favour large visible defects over subtle ones. Compare AUROC between defect types if you have pixel-level annotations.
- **Training imbalance:** ViT training uses many more good than defective samples. AUROC is robust to this, but accuracy is not—always prioritise AUROC.

### 7.3 Ethical & Societal Implications
- **Safety:** Missed defects in bottles, pills, or transistors can harm consumers. Recommend human-in-the-loop deployment.
- **Economic trade-off:** False positives waste good product; false negatives risk recalls. The cost asymmetry justifies conservative thresholds.
- **Transparency:** Document which categories perform poorly and why.

### 7.4 Suggested Extensions for Your Report
1. **Robustness analysis:** Test on the RobustAD benchmark (real acquisition shifts) or apply synthetic corruptions (brightness, blur, noise) to evaluate generalisation.
2. **Cross-category transfer:** Train ViT on one category, test on another—measure how much the learned representations transfer.
3. **Pixel-level segmentation:** Use the anomaly heatmaps from CLIP patch tokens or ViT attention for localisation, not just classification.
4. **Prompt engineering:** Test multiple prompt templates per category and ensemble the scores.

---
## 🚑 Troubleshooting Quick Reference

| Symptom | Fix |
| :--- | :--- |
| `NoMatchingPeftModuleError` | Re-run the LoRA cell – it auto-discovers target modules. |
| `CUDA OOM` | Lower `per_device_train_batch_size` to 8 or 4. |
| Training takes too long | Reduce `SELECTED_CATEGORIES` to 3-5 key categories. |
| `Loaded 0 images for 'X'` | Check that the category name matches exactly (case-sensitive). |
| Want to resume later | Set `SELECTED_CATEGORIES` to remaining categories and load saved adapters. |